#1. Data Loading & Understanding a) Load the dataset into Python. b) Display the first 5 rows. c) Check the shape of the dataset. d) Display the data types of each column.

```
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_curve, roc_auc_score
from sklearn.pipeline import Pipeline

# In Google Colab, upload loan_approval.xlsx when prompted
from google.colab import files
uploaded = files.upload()

df = pd.read_excel('loan_approval.xlsx')
display(df.head())
print("Dataset shape:", df.shape)
print("\\nData types:")
print(df.dtypes)
print("\\nSummary statistics:")
display(df.describe(include='all'))
```


#2. Data Cleaning a) Check for missing values in the dataset. b) Handle missing values appropriately. c) Identify categorical columns present in the dataset.

```
print("Missing values before cleaning:")
print(df.isnull().sum())

# Numeric columns: fill missing values with median; categorical columns: fill with mode
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print("Categorical columns:", categorical_cols)
print("Missing values after cleaning:")
print(df.isnull().sum())
```

#3. Exploratory Data Analysis (EDA) a) Plot the distribution of the target variable. b) Analyze the relationship between one numerical feature and the target variable. c) Analyze the relationship between Years of Employment and the target variable.

```
# Target distribution
print(df['loan_approved'].value_counts())
sns.countplot(data=df, x='loan_approved')
plt.title('Distribution of Loan Approval')
plt.xlabel('Loan Approved (False=0, True=1)')
plt.ylabel('Count')
plt.show()

# Relationship between a numerical feature (credit_score) and target
sns.boxplot(data=df, x='loan_approved', y='credit_score')
plt.title('Credit Score vs Loan Approval')
plt.show()

# Years of employment vs target
sns.boxplot(data=df, x='loan_approved', y='years_employed')
plt.title('Years of Employment vs Loan Approval')
plt.show()

print(df.groupby('loan_approved')[['credit_score', 'years_employed']].mean())
```

#4. Outlier Detection and Treatment a) Detect outliers in numerical columns using the IQR method. b) Treat outliers using capping techniques

```
# Work on a copy; exclude the target from outlier treatment
model_df = df.drop(columns=['name', 'city']).copy()
model_df['loan_approved'] = model_df['loan_approved'].astype(int)

numeric_features = model_df.drop(columns=['loan_approved']).select_dtypes(include=np.number).columns.tolist()

outlier_bounds = {}
for col in numeric_features:
    q1 = model_df[col].quantile(0.25)
    q3 = model_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_bounds[col] = (lower, upper)
    n_outliers = ((model_df[col] < lower) | (model_df[col] > upper)).sum()
    print(f'{col}: lower={lower:.2f}, upper={upper:.2f}, outliers={n_outliers}')
    model_df[col] = model_df[col].clip(lower, upper)
 ```

#5. Convert target variables into numerical format suitable for model building and drop unnecessary columns.

```
# loan_approved is converted from Boolean to 0/1 above.
# name and city are excluded because they are identifiers/high-cardinality text fields.
display(model_df.head())
print(model_df['loan_approved'].value_counts())
```


#6. Feature Selection and Data Splitting a) Separate independent variables (X) and dependent variable (y). b) Split the dataset into training and testing sets.

```
X = model_df.drop(columns=['loan_approved'])
y = model_df['loan_approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

```

#7. Apply feature scaling to the dataset using StandardScaler.

```
# Scaling is fitted only on training data through the Pipeline to avoid data leakage.
model = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic_regression', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print('First 10 predictions:', y_pred[:10])

```

#8. Logistic Regression Model Building a) Train a Logistic Regression model. b) Predict the output for test data.

```
# Scaling is fitted only on training data through the Pipeline to avoid data leakage.
model = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic_regression', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print('First 10 predictions:', y_pred[:10])

```

#9. Model Evaluation – Confusion Matrix a) Generate the confusion matrix. b) Interpret the results.

```
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:\\n', cm)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\\nClassification report:\\n', classification_report(y_test, y_pred, zero_division=0))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Not Approved', 'Predicted Approved'],
            yticklabels=['Actual Not Approved', 'Actual Approved'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

```

#10. Model Evaluation – ROC Curve & AUC a) Plot the ROC curve. b) Calculate the AUC score.

```
lr = model.named_steps['logistic_regression']
feature_names = X.columns

coef_table = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lr.coef_[0],
    'Odds_Ratio': np.exp(lr.coef_[0])
})
display(coef_table.sort_values('Coefficient', ascending=False))

print('Intercept:', lr.intercept_[0])
print('Logistic model: log(p / (1-p)) = intercept + sum(coefficient_i * standardized_feature_i)')
print('Here p is the probability that loan_approved = 1.')
print('Positive coefficients increase approval odds; negative coefficients decrease approval odds, holding other features constant.')
print('Odds ratio above 1 indicates increased odds per one-standard-deviation increase; below 1 indicates decreased odds.')

```